# 🚇 AI That Predicts Which Metro Evacuation Route Will Flood
## Civil Engineering + Deep Learning with an LSTM

A normal evacuation route may look safe now and flood while passengers are using it. This notebook predicts **water depth five minutes ahead** for two routes and recommends the safer exit.

> This is an educational decision-support demonstration. The safety bands below are project assumptions, not universal evacuation standards. A real deployment requires site-specific hydraulic analysis, approved operating procedures, sensor redundancy, validation, and human command.

👉 **Open the interactive companion:** [https://metro-evacuation-flood.streamlit.app](https://metro-evacuation-flood.streamlit.app/?stage=start)

## The complete system

Five measurements each minute → previous ten minutes → LSTM → route depth at +5 minutes → project safety band → compare Route A and Route B → recommend exit.

| Input | Unit | Meaning |
|---|---:|---|
| Rainfall intensity | mm/hr | Water supplied by the storm |
| Entrance water level | cm | Water near the station entrance |
| Tunnel water level | cm | Water entering through the tunnel |
| Drainage flow | L/s | Water removed by drains |
| Current route water level | cm | Water already on the route |

**Output:** route water depth in centimetres, five minutes later.

## Interactive learning journey

- [A Station During Heavy Rain](https://metro-evacuation-flood.streamlit.app/?stage=station) — Why Prediction Is Needed
- [Five Flood Measurements](https://metro-evacuation-flood.streamlit.app/?stage=inputs) — Input Features
- [Checking the Instrument Log](https://metro-evacuation-flood.streamlit.app/?stage=prepare) — Cleaning and Normalization
- [The Previous Ten Minutes](https://metro-evacuation-flood.streamlit.app/?stage=sequences) — Time Sequences
- [Remembering Accumulation](https://metro-evacuation-flood.streamlit.app/?stage=lstm) — LSTM Memory
- [Learning From Recorded Storms](https://metro-evacuation-flood.streamlit.app/?stage=training) — Model Training
- [Water Depth Five Minutes Ahead](https://metro-evacuation-flood.streamlit.app/?stage=forecast) — Regression Forecast
- [Project Safety Bands](https://metro-evacuation-flood.streamlit.app/?stage=thresholds) — Decision Thresholds
- [Route A Versus Route B](https://metro-evacuation-flood.streamlit.app/?stage=routes) — Decision Fusion
- [The Engineering Audit](https://metro-evacuation-flood.streamlit.app/?stage=audit) — Forecast Evaluation

In [ ]:
# Colab setup: uncomment only if your runtime is missing a package.
# !pip -q install tensorflow scikit-learn pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
FEATURES = ["rainfall_mm_hr", "entrance_water_cm", "tunnel_water_cm", "drainage_l_s", "route_water_cm"]
LOOKBACK, HORIZON = 10, 5
print("Ready: 5 inputs, 10-minute history, 5-minute forecast")

---
# 1. A Station During Heavy Rain
### Phase 1 of 6 · The Station

## Part 1 · On the metro
Passengers are on an underground platform with two evacuation routes. Both routes can look usable while water is entering through the station entrance and tunnel.

## Part 2 · The engineering challenge
Evacuation takes time. A route selected from its current water depth can become unsafe while passengers are still moving through it.

## Part 3 · Where the AI comes in
Forecast the depth five minutes ahead, early enough to redirect passengers before the route deteriorates.

**Civil Engineering:** A Station During Heavy Rain → **AI:** Why Prediction Is Needed → **Technical mechanism:** `Current safety is not future safety`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=station](https://metro-evacuation-flood.streamlit.app/?stage=station)

## Part 4 · Technical explanation

At 10:03, Route A can contain only 4 cm of water and appear usable. If rainfall, entrance inflow, and tunnel inflow are rising while drainage is weakening, its condition at 10:08 is the relevant engineering question.

```
             EXIT A
                ↑
           ROUTE A
                ↑
PLATFORM ───────┤
                ↓
           ROUTE B
                ↓
             EXIT B
```

## Part 5 · What you just built

**In the notebook:** Define the two-route station and the five-minute forecasting target.

**Takeaway:** Choose a route for the conditions passengers will meet, not only those visible now.

[Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Five Flood Measurements](https://metro-evacuation-flood.streamlit.app/?stage=inputs) ▶

---
# 2. Five Flood Measurements
### Phase 2 of 6 · Reading the Water

## Part 1 · On the metro
Rain supplies water; entrance and tunnel levels describe where it is arriving; drains remove it; the route level shows what has already accumulated.

## Part 2 · The engineering challenge
No single measurement describes the balance. Heavy rain may be manageable with strong drainage, while moderate rain can be dangerous when the drains are overloaded.

## Part 3 · Where the AI comes in
The model receives only these five measurements so every prediction remains explainable in engineering terms.

**Civil Engineering:** Five Flood Measurements → **AI:** Input Features → **Technical mechanism:** `rainfall, entrance, tunnel, drainage, route depth`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=inputs](https://metro-evacuation-flood.streamlit.app/?stage=inputs)

## Part 4 · Technical explanation

In [ ]:
def generate_route(route, minutes=1440, seed=42):
    rng = np.random.default_rng(seed)
    t = np.arange(minutes)
    # Several smooth storm cells plus instrument noise.
    rain = np.zeros(minutes)
    for centre, width, peak in [(260,55,72),(690,80,105),(1120,65,82)]:
        rain += peak*np.exp(-0.5*((t-centre)/width)**2)
    rain = np.clip(rain + rng.normal(0,2.2,minutes), 0, None)
    entrance = np.zeros(minutes); tunnel = np.zeros(minutes); drainage = np.zeros(minutes); depth = np.zeros(minutes)
    bias = 1.0 if route == "A" else .68
    protection = 1.0 if route == "A" else 1.25
    for i in range(1, minutes):
        entrance[i] = max(0, .86*entrance[i-1] + .055*rain[i-2] + rng.normal(0,.35))*bias
        tunnel[i] = max(0, .91*tunnel[i-1] + .026*rain[i-3] + rng.normal(0,.20))*bias
        drainage[i] = np.clip((17-.075*rain[i]+rng.normal(0,.6))*protection, 3, 24)
        inflow = .050*rain[i] + .13*entrance[i] + .18*tunnel[i]
        depth[i] = max(0, .94*depth[i-1] + .16*inflow - .115*drainage[i] + rng.normal(0,.16))
    return pd.DataFrame({"minute":t,"route":route,"rainfall_mm_hr":rain,
                         "entrance_water_cm":entrance,"tunnel_water_cm":tunnel,
                         "drainage_l_s":drainage,"route_water_cm":depth})

data = pd.concat([generate_route("A", seed=42), generate_route("B", seed=84)], ignore_index=True)
print(data.shape)
data.head()

In [ ]:
fig, ax = plt.subplots(2,1,figsize=(13,7),sharex=True)
for route, colour in [("A","crimson"),("B","seagreen")]:
    d=data[data.route==route]
    ax[0].plot(d.minute,d.rainfall_mm_hr,label=f"Route {route}",alpha=.8,color=colour)
    ax[1].plot(d.minute,d.route_water_cm,label=f"Route {route}",color=colour)
ax[0].set_ylabel("Rainfall (mm/hr)"); ax[1].set_ylabel("Route depth (cm)"); ax[1].set_xlabel("Minute")
for a in ax: a.grid(alpha=.2); a.legend()
plt.tight_layout(); plt.show()

## Part 5 · What you just built

**In the notebook:** Generate one-minute measurements for Routes A and B.

**Takeaway:** Flood risk is the balance between water entering, water leaving, and water already stored.

◀ [Previous: A Station During Heavy Rain](https://metro-evacuation-flood.streamlit.app/?stage=station) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Checking the Instrument Log](https://metro-evacuation-flood.streamlit.app/?stage=prepare) ▶

---
# 3. Checking the Instrument Log
### Phase 2 of 6 · Reading the Water

## Part 1 · On the metro
A level sensor may miss a reading and every instrument uses different units. Engineers check the log before trusting it.

## Part 2 · The engineering challenge
An LSTM cannot interpret a missing value, and rainfall in mm/hr can dominate route depth in centimetres simply because its numbers are larger.

## Part 3 · Where the AI comes in
Interpolate short gaps and scale all five inputs using statistics learned from training data only.

**Civil Engineering:** Checking the Instrument Log → **AI:** Cleaning and Normalization → **Technical mechanism:** `interpolate gaps; fit scaler on training data`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=prepare](https://metro-evacuation-flood.streamlit.app/?stage=prepare)

## Part 4 · Technical explanation

In [ ]:
# Demonstrate realistic short sensor gaps, then repair them per route.
dirty = data.copy()
rng = np.random.default_rng(7)
gap_rows = rng.choice(dirty.index, size=18, replace=False)
dirty.loc[gap_rows, "tunnel_water_cm"] = np.nan
print("Missing before cleaning:", dirty[FEATURES].isna().sum().sum())
clean = dirty.sort_values(["route","minute"]).copy()
clean[FEATURES] = clean.groupby("route")[FEATURES].transform(lambda s: s.interpolate().bfill().ffill())
print("Missing after cleaning :", clean[FEATURES].isna().sum().sum())

## Part 5 · What you just built

**In the notebook:** Clean the synthetic log and apply MinMaxScaler without test leakage.

**Takeaway:** A clean, consistently scaled sequence is part of the model, not clerical preparation.

◀ [Previous: Five Flood Measurements](https://metro-evacuation-flood.streamlit.app/?stage=inputs) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Previous Ten Minutes](https://metro-evacuation-flood.streamlit.app/?stage=sequences) ▶

---
# 4. The Previous Ten Minutes
### Phase 3 of 6 · Remembering the Storm

## Part 1 · On the metro
A rising route level means something different after ten minutes of heavy rain than after ten minutes of recovery.

## Part 2 · The engineering challenge
A row-by-row model sees the present snapshot but loses the direction and persistence of the storm.

## Part 3 · Where the AI comes in
Each training example contains the previous ten one-minute readings and the route depth five minutes after the final reading.

**Civil Engineering:** The Previous Ten Minutes → **AI:** Time Sequences → **Technical mechanism:** `10 x 5 input window -> depth at t+5`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=sequences](https://metro-evacuation-flood.streamlit.app/?stage=sequences)

## Part 4 · Technical explanation

In [ ]:
def make_raw_sequences(frame, lookback=LOOKBACK, horizon=HORIZON):
    X, y, routes, target_minutes = [], [], [], []
    for route, d in frame.groupby("route", sort=False):
        d=d.sort_values("minute").reset_index(drop=True)
        values=d[FEATURES].to_numpy(dtype=np.float32)
        for end in range(lookback, len(d)-horizon):
            X.append(values[end-lookback:end])
            y.append(values[end+horizon-1, FEATURES.index("route_water_cm")])
            routes.append(route); target_minutes.append(int(d.loc[end+horizon-1,"minute"]))
    return np.array(X), np.array(y), np.array(routes), np.array(target_minutes)

X_raw,y_raw,route_id,target_minute=make_raw_sequences(clean)
print("X:",X_raw.shape,"= examples × 10 minutes × 5 inputs")
print("y:",y_raw.shape,"= depth 5 minutes later")

## Part 5 · What you just built

**In the notebook:** Convert the continuous log into sliding windows and future-depth targets.

**Takeaway:** Flooding is a process, so the model must see a history.

◀ [Previous: Checking the Instrument Log](https://metro-evacuation-flood.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Remembering Accumulation](https://metro-evacuation-flood.streamlit.app/?stage=lstm) ▶

---
# 5. Remembering Accumulation
### Phase 3 of 6 · Remembering the Storm

## Part 1 · On the metro
Water on a route is stored history: earlier inflow still matters until drainage removes it.

## Part 2 · The engineering challenge
Recent measurements are not equally useful. The model must retain a sustained rise and forget noise or an old shower that has already drained.

## Part 3 · Where the AI comes in
An LSTM learns gates that decide what history to keep, update, and use for the forecast.

**Civil Engineering:** Remembering Accumulation → **AI:** LSTM Memory → **Technical mechanism:** `gates retain, update, and expose useful state`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=lstm](https://metro-evacuation-flood.streamlit.app/?stage=lstm)

## Part 4 · Technical explanation

In [ ]:
# Time-ordered split: earlier storm history trains; later history tests.
train_mask = target_minute < 950
val_mask = (target_minute >= 950) & (target_minute < 1160)
test_mask = target_minute >= 1160

# Fit scaling on training measurements only.
x_scaler=MinMaxScaler().fit(X_raw[train_mask].reshape(-1,len(FEATURES)))
y_scaler=MinMaxScaler().fit(y_raw[train_mask].reshape(-1,1))
def sx(x): return x_scaler.transform(x.reshape(-1,len(FEATURES))).reshape(x.shape)
X_scaled=sx(X_raw); y_scaled=y_scaler.transform(y_raw.reshape(-1,1))

model=Sequential([
    Input(shape=(LOOKBACK,len(FEATURES))),
    LSTM(32),
    Dropout(.15),
    Dense(16,activation="relu"),
    Dense(1)
])
model.compile(optimizer="adam",loss="mae",metrics=["mae"])
model.summary()

## Part 5 · What you just built

**In the notebook:** Build an LSTM(32) followed by Dense layers and one regression output.

**Takeaway:** LSTM memory represents the evolving balance of accumulation and drainage.

◀ [Previous: The Previous Ten Minutes](https://metro-evacuation-flood.streamlit.app/?stage=sequences) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Learning From Recorded Storms](https://metro-evacuation-flood.streamlit.app/?stage=training) ▶

---
# 6. Learning From Recorded Storms
### Phase 4 of 6 · Forecasting Five Minutes

## Part 1 · On the metro
Recorded storm events show how route water actually responded to different rainfall and drainage conditions.

## Part 2 · The engineering challenge
The model must learn the relationship without memorising one storm or training indefinitely on noise.

## Part 3 · Where the AI comes in
Train on earlier sequences, validate on later ones, and stop when validation error no longer improves.

**Civil Engineering:** Learning From Recorded Storms → **AI:** Model Training → **Technical mechanism:** `MAE loss + Adam optimizer + early stopping`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=training](https://metro-evacuation-flood.streamlit.app/?stage=training)

## Part 4 · Technical explanation

In [ ]:
early=EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True)
history=model.fit(X_scaled[train_mask],y_scaled[train_mask],
                  validation_data=(X_scaled[val_mask],y_scaled[val_mask]),
                  epochs=60,batch_size=64,callbacks=[early],verbose=0)
print("Epochs run:",len(history.history["loss"]))
plt.figure(figsize=(9,4)); plt.plot(history.history["loss"],label="train MAE")
plt.plot(history.history["val_loss"],label="validation MAE"); plt.xlabel("Epoch"); plt.ylabel("Scaled MAE")
plt.grid(alpha=.2); plt.legend(); plt.show()

## Part 5 · What you just built

**In the notebook:** Train the LSTM with MAE loss and EarlyStopping.

**Takeaway:** Training adjusts the forecast; validation tells us when adjustment stops helping.

◀ [Previous: Remembering Accumulation](https://metro-evacuation-flood.streamlit.app/?stage=lstm) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Water Depth Five Minutes Ahead](https://metro-evacuation-flood.streamlit.app/?stage=forecast) ▶

---
# 7. Water Depth Five Minutes Ahead
### Phase 4 of 6 · Forecasting Five Minutes

## Part 1 · On the metro
The control room needs an estimated depth, not merely a vague danger label.

## Part 2 · The engineering challenge
A label hides how close the route is to a decision boundary and prevents engineers from applying a different project rule.

## Part 3 · Where the AI comes in
The LSTM outputs a continuous depth in centimetres; a separate project assumption converts it into a safety band.

**Civil Engineering:** Water Depth Five Minutes Ahead → **AI:** Regression Forecast → **Technical mechanism:** `one continuous output in centimetres`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=forecast](https://metro-evacuation-flood.streamlit.app/?stage=forecast)

## Part 4 · Technical explanation

In [ ]:
pred_scaled=model.predict(X_scaled[test_mask],verbose=0)
pred_cm=y_scaler.inverse_transform(pred_scaled).ravel()
actual_cm=y_raw[test_mask]
mae=mean_absolute_error(actual_cm,pred_cm)
rmse=mean_squared_error(actual_cm,pred_cm)**0.5
print(f"Test MAE : {mae:.2f} cm")
print(f"Test RMSE: {rmse:.2f} cm")

n=min(260,len(pred_cm)); plt.figure(figsize=(13,4))
plt.plot(actual_cm[:n],label="Actual",color="black",linewidth=2)
plt.plot(pred_cm[:n],label="LSTM predicted",color="deepskyblue")
plt.ylabel("Route water depth (cm)"); plt.xlabel("Successive test sequences")
plt.grid(alpha=.2); plt.legend(); plt.show()

## Part 5 · What you just built

**In the notebook:** Inverse-transform the prediction and plot actual versus predicted depth.

**Takeaway:** Predict the physical quantity first; apply the operational rule second.

◀ [Previous: Learning From Recorded Storms](https://metro-evacuation-flood.streamlit.app/?stage=training) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Project Safety Bands](https://metro-evacuation-flood.streamlit.app/?stage=thresholds) ▶

---
# 8. Project Safety Bands
### Phase 5 of 6 · Choosing the Exit

## Part 1 · On the metro
The demonstration needs a clear rule for when a route should no longer be selected.

## Part 2 · The engineering challenge
There is no universal evacuation-depth standard established by this educational model; presenting one as universal would be unsafe.

## Part 3 · Where the AI comes in
Use explicit project assumptions: below 10 cm SAFE, 10-20 cm WARNING, above 20 cm UNSAFE. A real project requires authority approval and site-specific analysis.

**Civil Engineering:** Project Safety Bands → **AI:** Decision Thresholds → **Technical mechanism:** `<10 safe; 10-20 warning; >20 unsafe`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=thresholds](https://metro-evacuation-flood.streamlit.app/?stage=thresholds)

## Part 4 · Technical explanation

In [ ]:
def safety_band(depth_cm):
    # EDUCATIONAL PROJECT ASSUMPTION — not a universal evacuation standard.
    if depth_cm < 10: return "SAFE"
    if depth_cm <= 20: return "WARNING"
    return "UNSAFE"

for depth in [6,15,28]: print(depth,"cm ->",safety_band(depth))

## Part 5 · What you just built

**In the notebook:** Implement a transparent status function separate from the LSTM.

**Takeaway:** Thresholds are declared project assumptions, not facts learned by the model.

◀ [Previous: Water Depth Five Minutes Ahead](https://metro-evacuation-flood.streamlit.app/?stage=forecast) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Route A Versus Route B](https://metro-evacuation-flood.streamlit.app/?stage=routes) ▶

---
# 9. Route A Versus Route B
### Phase 5 of 6 · Choosing the Exit

## Part 1 · On the metro
The platform has two exits. Route A may receive entrance water while Route B remains protected by stronger drainage.

## Part 2 · The engineering challenge
A prediction becomes useful only when it changes the evacuation decision before conditions become dangerous.

## Part 3 · Where the AI comes in
Forecast both routes with the same five inputs, classify each using the assumed bands, and recommend the safer usable route.

**Civil Engineering:** Route A Versus Route B → **AI:** Decision Fusion → **Technical mechanism:** `forecast both routes; choose lowest acceptable depth`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=routes](https://metro-evacuation-flood.streamlit.app/?stage=routes)

## Part 4 · Technical explanation

In [ ]:
def latest_forecast(route):
    candidates=np.where(test_mask & (route_id==route))[0]
    idx=candidates[-1]
    prediction=y_scaler.inverse_transform(model.predict(X_scaled[idx:idx+1],verbose=0)).item()
    return prediction

route_a=latest_forecast("A"); route_b=latest_forecast("B")
def recommend(a,b):
    usable=[(d,r) for d,r in [(a,"A"),(b,"B")] if safety_band(d)!="UNSAFE"]
    if not usable: return "NO MODEL-APPROVED ROUTE — invoke emergency procedure"
    return "ROUTE "+min(usable)[1]

print(f"Route A after 5 min: {route_a:.1f} cm — {safety_band(route_a)}")
print(f"Route B after 5 min: {route_b:.1f} cm — {safety_band(route_b)}")
print("AI RECOMMENDATION:",recommend(route_a,route_b))

```
             EXIT A
               ↑
          ROUTE A ❌
        Predicted: 28 cm
               ↑
            PLATFORM
               ↓
          ROUTE B ✓
         Predicted: 6 cm
               ↓
             EXIT B

AI RECOMMENDATION: EVACUATE USING ROUTE B
```

## Part 5 · What you just built

**In the notebook:** Compare Route A and Route B predictions and print the recommendation.

**Takeaway:** The final output is an actionable route recommendation, supported by two forecasts.

◀ [Previous: Project Safety Bands](https://metro-evacuation-flood.streamlit.app/?stage=thresholds) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Engineering Audit](https://metro-evacuation-flood.streamlit.app/?stage=audit) ▶

---
# 10. The Engineering Audit
### Phase 6 of 6 · Engineering Audit

## Part 1 · On the metro
Before an evacuation system is trusted, engineers ask how far forecasts miss and whether those misses change the route decision.

## Part 2 · The engineering challenge
A low average error can still hide dangerous cases where an unsafe route is predicted as safe.

## Part 3 · Where the AI comes in
Evaluate on later, unseen storm periods using MAE and RMSE, then count safety-band and unsafe-as-safe errors separately.

**Civil Engineering:** The Engineering Audit → **AI:** Forecast Evaluation → **Technical mechanism:** `MAE, RMSE, threshold errors, time-ordered test`

> 🎬 **See this illustrated and interactive:** [https://metro-evacuation-flood.streamlit.app/?stage=audit](https://metro-evacuation-flood.streamlit.app/?stage=audit)

## Part 4 · Technical explanation

In [ ]:
def bands(values): return np.array([safety_band(v) for v in values])
actual_band=bands(actual_cm); predicted_band=bands(pred_cm)
band_accuracy=(actual_band==predicted_band).mean()
unsafe_as_safe=((actual_band=="UNSAFE") & (predicted_band=="SAFE")).sum()
print(f"MAE                 : {mae:.2f} cm")
print(f"RMSE                : {rmse:.2f} cm")
print(f"Safety-band accuracy: {band_accuracy:.1%}")
print(f"UNSAFE predicted SAFE: {unsafe_as_safe}")
print("\nA real project would also test sensor failures, rare extreme storms, uncertainty, hydraulic-model consistency, latency, alarms, and evacuation drills.")

## Part 5 · What you just built

**In the notebook:** Report regression metrics and decision-confusion counts on the test period.

**Takeaway:** Audit the consequence of error, not only its average size.

◀ [Previous: Route A Versus Route B](https://metro-evacuation-flood.streamlit.app/?stage=routes) &nbsp;|&nbsp; [Project overview](https://metro-evacuation-flood.streamlit.app/?stage=start)

---
# Final engineering conclusion

The LSTM predicts a physical quantity—route water depth five minutes ahead—from the previous ten minutes of five measurements. A separate, transparent project rule assigns a safety band. Forecasts for Routes A and B are compared, and the control room receives an actionable recommendation.

The system does **not** replace evacuation authorities or guarantee safety. Its meaningful contribution is earlier warning: it can reject a route that looks safe now but is forecast to flood before passengers complete the evacuation.